# Qwen3.5-0.8B × TinyCeNN PDelta3-GDN2-CLVR — Sequential Full-Attention Replacement

This experiment applies the **SmolLM2 PDelta3/GDN2 + Local32 + CLVR** idea to `Qwen/Qwen3.5-0.8B`.

Qwen3.5 is already a hybrid model: 24 text layers use a **3:1 pattern of native Gated DeltaNet linear attention and full attention**. Therefore this notebook does **not** replace the 18 native linear-attention layers. It targets only the 6 full-attention layers (`3, 7, 11, 15, 19, 23`) one by one.

The pilot replaces the first **3 full-attention layers**. If their held-out quality gates pass, set `TARGET_FULL_LAYERS = 6` and rerun with `RESUME=True`.

The run is text-only (`Qwen3_5ForCausalLM`); multimodal/vision evaluation comes after the text backbone is validated.


In [ ]:
import os, sys, pathlib, subprocess

# Public model: explicitly disable Hugging Face implicit-token lookup.
# This avoids the Colab warning/time-out caused by trying to read HF_TOKEN
# from the Secrets vault when the notebook is not running with that secret.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR),
        "transformers==4.57.6",
        "datasets>=3,<5",
        "huggingface_hub>=0.34,<2",
        "safetensors",
        "pandas",
        "matplotlib",
    ],
    check=True,
)

for p in (REPO_DIR / "src", REPO_DIR, REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
import transformers
from transformers import Qwen3_5ForCausalLM

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/TinyCeNN-LM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("persistent root:", DRIVE_ROOT)


## Settings

The pilot keeps the same successful SmolLM2 architecture (`feature_dim=96`, `Local32`) but uses a more conservative learning rate because Qwen3.5-0.8B is much larger and already has a strong hybrid token mixer.

`TARGET_FULL_LAYERS=3` means actual Qwen text layers **3, 7, and 11**. Increase it to `6` only after the pilot behaves well.


In [ ]:
BASE_MODEL = "Qwen/Qwen3.5-0.8B"

# Architecture
FEATURE_DIM = 96
LOCAL_WINDOW = 32
CHUNK_SIZE = 32
CONV_KERNEL = 4
STATE_DTYPE = "fp16"
LOCAL_GATE_INIT = 0.72
WARM_START_PREVIOUS_CORE = True

# Qwen3.5-0.8B has full attention at 3,7,11,15,19,23.
TARGET_FULL_LAYERS = 3

# Start smaller than the SmolLM2 run: Qwen has 248k vocabulary and ~0.8B params.
CONTEXT_LENGTH = 128
PROBE_CONTEXT = 128
PROBE_BLOCKS = 6
SEED = 2026

MIN_LAYER_STEPS = 60
MAX_LAYER_STEPS = 250
CHECK_EVERY = 25
LAYER_LR = 2e-4
QKV_LR_SCALE = 0.10
TRAIN_QKV = True
TEMPERATURE = 1.5

FUNCTIONAL_WEIGHT = 0.30
KL_WEIGHT = 1.00
CE_WEIGHT = 0.08
COSINE_WEIGHT = 0.20
LOCAL_GATE_PENALTY = 0.001

# Rescue rounds emphasize language-model quality.
RESCUE_LR_SCALE = 0.50
RESCUE_FUNCTIONAL_WEIGHT = 0.15
RESCUE_KL_WEIGHT = 1.50
RESCUE_CE_WEIGHT = 0.12

# Same strict scientific gates used in the successful Smol experiment.
ACCEPT_NMSE = 0.15
ACCEPT_COSINE = 0.94
ACCEPT_INCREMENTAL_DELTA_NLL = 0.015
ACCEPT_CUMULATIVE_DELTA_NLL = 0.05
STRICT_ACCEPTANCE = True

RESUME = True
MAX_ROUNDS_PER_RUN = 2
MAX_RUNTIME_MINUTES = 240

OUTPUT_DIR = DRIVE_ROOT / (
    f"qwen35-0.8b-pdelta3-gdn2-clvr-local{LOCAL_WINDOW}-f{FEATURE_DIM}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("output:", OUTPUT_DIR)
print("target number of full-attention layers:", TARGET_FULL_LAYERS)


## Run / resume

The trainer automatically discovers Qwen3.5's full-attention positions and replaces only that prefix. It exits successfully when the current layer needs more training; rerun this same cell to continue from the **best saved quality checkpoint**.


In [ ]:
import signal, json, time

cmd = [
    sys.executable, "-u",
    str(REPO_DIR / "scripts" / "run_qwen35_pdelta3_clvr_sequential_colab.py"),
    "--base-model", BASE_MODEL,
    "--output-dir", str(OUTPUT_DIR),
    "--feature-dim", str(FEATURE_DIM),
    "--local-window", str(LOCAL_WINDOW),
    "--chunk-size", str(CHUNK_SIZE),
    "--conv-kernel", str(CONV_KERNEL),
    "--state-dtype", STATE_DTYPE,
    "--local-gate-init", str(LOCAL_GATE_INIT),
    "--target-full-layers", str(TARGET_FULL_LAYERS),
    "--context-length", str(CONTEXT_LENGTH),
    "--probe-context", str(PROBE_CONTEXT),
    "--probe-blocks", str(PROBE_BLOCKS),
    "--seed", str(SEED),
    "--min-layer-steps", str(MIN_LAYER_STEPS),
    "--max-layer-steps", str(MAX_LAYER_STEPS),
    "--check-every", str(CHECK_EVERY),
    "--layer-lr", str(LAYER_LR),
    "--qkv-lr-scale", str(QKV_LR_SCALE),
    "--temperature", str(TEMPERATURE),
    "--functional-weight", str(FUNCTIONAL_WEIGHT),
    "--kl-weight", str(KL_WEIGHT),
    "--ce-weight", str(CE_WEIGHT),
    "--cosine-weight", str(COSINE_WEIGHT),
    "--local-gate-penalty", str(LOCAL_GATE_PENALTY),
    "--rescue-lr-scale", str(RESCUE_LR_SCALE),
    "--rescue-functional-weight", str(RESCUE_FUNCTIONAL_WEIGHT),
    "--rescue-kl-weight", str(RESCUE_KL_WEIGHT),
    "--rescue-ce-weight", str(RESCUE_CE_WEIGHT),
    "--accept-nmse", str(ACCEPT_NMSE),
    "--accept-cosine", str(ACCEPT_COSINE),
    "--accept-incremental-delta-nll", str(ACCEPT_INCREMENTAL_DELTA_NLL),
    "--accept-cumulative-delta-nll", str(ACCEPT_CUMULATIVE_DELTA_NLL),
    "--max-runtime-minutes", str(MAX_RUNTIME_MINUTES),
]
cmd.append("--warm-start-previous-core" if WARM_START_PREVIOUS_CORE else "--no-warm-start-previous-core")
cmd.append("--train-qkv" if TRAIN_QKV else "--no-train-qkv")
cmd.append("--strict-acceptance" if STRICT_ACCEPTANCE else "--no-strict-acceptance")
cmd.append("--resume" if RESUME else "--no-resume")

log_path = OUTPUT_DIR / "last_colab_run.log"
print(" ".join(cmd))
print("Log:", log_path)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["SEQUENTIAL_MAX_ROUNDS_PER_RUN"] = str(MAX_ROUNDS_PER_RUN)
env["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

interrupted = False
with log_path.open("w", encoding="utf-8") as log:
    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert p.stdout is not None
    try:
        for line in p.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        rc = p.wait()
    except KeyboardInterrupt:
        interrupted = True
        print("\nInterrupt requested. Asking the trainer to stop cleanly...")
        p.send_signal(signal.SIGINT)
        try:
            rc = p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            p.terminate()
            rc = p.wait()
        print("Persistent Drive checkpoints remain usable.")

print("Process return code:", rc)
if rc != 0 and not interrupted:
    raise RuntimeError(f"Trainer failed with return code {rc}. See {log_path}.")


## Status / layer-by-layer trend

In [ ]:
import json, pathlib
import pandas as pd

status_path = OUTPUT_DIR / "qwen35_run_status.json"
progress_path = OUTPUT_DIR / "qwen35_progress.json"
in_progress_path = OUTPUT_DIR / "qwen35_in_progress.json"

for path in (status_path, progress_path, in_progress_path):
    if path.exists():
        print("\n###", path.name)
        print(path.read_text()[:12000])

reports = []
if progress_path.exists():
    reports.extend(json.loads(progress_path.read_text()).get("reports", []))
if in_progress_path.exists():
    reports = json.loads(in_progress_path.read_text()).get("reports", reports)

if reports:
    df = pd.DataFrame(reports)
    cols = [c for c in ["layer","round","step","accepted","nmse","cosine","incremental_delta_nll","cumulative_delta_nll","local_gate_mean"] if c in df.columns]
    display(df[cols].tail(30))


## Prompt smoke test of the accepted prefix

This loads only the **accepted** Qwen full-attention replacements. It first generates baseline completions, frees the baseline model, then loads the candidate, so the comparison is safer on a T4.


In [ ]:
import gc, json, torch, pathlib, sys
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

sys.path.insert(0, str(REPO_DIR / "scripts"))
from train_qwen35_pdelta3_clvr_sequential import QwenPDelta3CLVRConfig, replace_full_attention_layers

progress_pt = OUTPUT_DIR / "qwen35_progress.pt"
if not progress_pt.exists():
    raise FileNotFoundError("No accepted Qwen3.5 replacement yet. Run/resume training until at least one full-attention layer passes.")

payload = torch.load(progress_pt, map_location="cpu", weights_only=False)
accepted = [int(x) for x in payload["accepted_full_attention_layers"]]
cfg = QwenPDelta3CLVRConfig.from_dict(payload["config"])
print("Accepted Qwen3.5 full-attention layers:", accepted)
print("Replacement config:", cfg.to_dict())

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=False, use_fast=True)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

prompts = [
    "The future of small language models is",
    "Artificial intelligence can help scientists by",
    "A good software architecture should",
    "The capital of Austria is",
    "Once upon a time, a small robot",
]

@torch.no_grad()
def generate_all(model):
    model.eval(); result = {}
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=False, pad_token_id=tokenizer.eos_token_id)
        result[prompt] = tokenizer.decode(out[0], skip_special_tokens=True)
    return result

print("Loading baseline...")
baseline = Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype, token=False, attn_implementation="eager").to(device)
baseline.config.use_cache = False
baseline_text = generate_all(baseline)
del baseline; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Loading accepted PDelta3-CLVR prefix...")
candidate = Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype, token=False, attn_implementation="eager").to(device)
candidate.config.use_cache = False
replace_full_attention_layers(candidate, cfg, accepted)
incompatible = candidate.load_state_dict(payload["attention_state"], strict=False)
missing = [k for k in incompatible.missing_keys if any(k.startswith(f"model.layers.{i}.self_attn.") for i in accepted)]
if missing: raise RuntimeError(f"Accepted checkpoint missing keys: {missing[:8]}")
candidate_text = generate_all(candidate)

rows = []
for prompt in prompts:
    print("\n" + "=" * 100); print("PROMPT:", prompt)
    print("\nBASELINE:\n", baseline_text[prompt]); print("\nPDELTA3-CLVR ACCEPTED PREFIX:\n", candidate_text[prompt])
    rows.append({"prompt":prompt,"baseline":baseline_text[prompt],"pdelta3_clvr":candidate_text[prompt]})

smoke_path = OUTPUT_DIR / "qwen35_prompt_smoke_test.json"
smoke_path.write_text(json.dumps({"base_model":BASE_MODEL,"accepted_full_attention_layers":accepted,"examples":rows}, indent=2), encoding="utf-8")
print("\nSaved:", smoke_path)
del candidate; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## After the pilot

If the first 3 full-attention layers are accepted and the prompt/NLL results remain good, change:

```python
TARGET_FULL_LAYERS = 6
```

Keep `RESUME = True` and rerun. The trainer will preserve the accepted layers and continue with actual Qwen layers `15, 19, 23`.
